In [37]:
"""
Copyright (c) Nolichucky Associates 2024. All Rights Reserved.

This software is the confidential and proprietary information of Nolichucky Associates.
You shall not disclose such Confidential Information and shall use it only in accordance
 with the terms of the license agreement you entered into with Nolichucky Associates.

Unauthorized copying of this file, via any medium, is strictly prohibited.
Proprietary and confidential.

Project: Music Worcester Patron and Event Analytics

Author: Anthony Smith
Date: 2025

Merges patron analytics (Patrons.csv) with donor history (DonationsLatest.xlsx),
then classifies patrons into five mutually exclusive donor prospect tranches.

Inputs:
    Patrons.csv          - output of MWSalesSumm.ipynb
    DonationsLatest.xlsx - Salesforce donation export

Output:
    Donor_Classification.xlsx         - one sheet per tranche, plus unmatched donor review sheet
    Donor_Classification_Anon.xlsx    - same, with PII columns removed
    Donor_Classification_Summary.xlsx - top-N summary for non-technical staff
    DonorNameMatchReview.xlsx          - fuzzy-match review for correcting Salesforce names
"""
import difflib
import logging
import os
import sys

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
default_data_dir = '/Users/antho/Documents/WPI-MW'
data_dir = input(f'Enter data directory [{default_data_dir}]: ').strip() or default_data_dir
os.chdir(data_dir)

patrons_file      = 'Patrons.csv'
donor_file        = 'DonationsLatest.xlsx'
output_file       = 'Donor_Classification.xlsx'
anon_file         = 'Donor_Classification_Anon.xlsx'
summary_file      = 'Donor_Classification_Summary.xlsx'
anon_summary_file = 'Donor_Classification_Summary_Anon.xlsx'
review_file        = 'DonorNameMatchReview.xlsx'
campaign_file      = 'Donor_Campaign.xlsx'
anon_campaign_file = 'Donor_Campaign_Anon.xlsx'

# Tranche thresholds — adjust here without touching logic below
MAJOR_DONOR_LIFETIME   = 5000  # Lifetime significant-gift threshold for Major Patrons
MAJOR_GIFT_MIN         = 500   # Minimum single gift to qualify as a significant donation
ACTIVE_DONATION_MONTHS = 18    # Months since last donation to be considered "active"
PRIME_RECENCY_MONTHS   = 12    # Ticket recency threshold for Prime Non-Donor Prospects
GROWTH_RECENCY_MONTHS  = 18    # Ticket recency threshold for Growth Prospects
SUMMARY_TOP_N          = 30    # Rows visible by default in summary; clear Rank filter to expand
MIN_SEASON_DONATION    = 100   # Minimum gift amount counted toward season determination (excludes ticket add-ons)

# Propensity score weights per tranche — (internal_column, weight) tuples, weights sum to 1.0
# Tune here; see _propensity_score for normalization details
T1_WEIGHTS = [('AverageDonation',  0.25), ('Regularity',        0.22),
              ('GrowthScore',       0.15), ('_DonorDueScore',    0.10),
              ('_IsChorus',         0.10), ('_PatronRank',       0.10),
              ('_SubscriberRank',   0.08)]
T2_WEIGHTS = [('GrowthScore',       0.23), ('AYM',               0.22),
              ('Regularity',        0.13), ('_DonorDueScore',    0.10),
              ('_IsChorus',         0.10), ('_PatronRank',       0.10),
              ('FullPriceRate',     0.08), ('_SubscriberRank',   0.04)]
T3_WEIGHTS = [('_DonorDueScore',    0.27), ('Regularity',        0.22),
              ('_LogDonationCount', 0.13), ('RFMScore',          0.10),
              ('_IsChorus',         0.10), ('_PatronRank',       0.10),
              ('_SubscriberRank',   0.08)]
T4_WEIGHTS = [('_DormantRecency',    0.27), ('RFMScore',           0.22),
              ('AverageDonation',    0.13), ('Regularity',         0.10),
              ('_IsChorus',          0.10), ('_PatronRank',        0.10),
              ('_SubscriberRank',    0.08)]
T5_WEIGHTS = [('AYM',                0.20), ('GrowthScore',        0.22),
              ('Regularity',         0.13), ('FullPriceRate',      0.12),
              ('_IsChorus',          0.10), ('_PatronRank',        0.10),
              ('Lifespan',           0.05), ('_SubscriberRank',    0.08)]
T6_WEIGHTS = [('AverageDonation',    0.37), ('RFMScore',           0.22),
              ('_DormantRecency',    0.13), ('_IsChorus',          0.10),
              ('_PatronRank',        0.10), ('_SubscriberRank',    0.08)]

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    c_handler = logging.StreamHandler(sys.stdout)
    c_handler.setFormatter(logging.Formatter('%(levelname)s - %(message)s'))
    c_handler.setLevel(logging.INFO)
    logger.addHandler(c_handler)

    f_handler = logging.FileHandler(os.path.join(data_dir, 'donor_classifier.log'), mode='w')
    f_handler.setFormatter(logging.Formatter('%(asctime)s - %(lineno)s - %(levelname)s - %(message)s'))
    f_handler.setLevel(logging.DEBUG)
    logger.addHandler(f_handler)


In [38]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
logger.info('Loading patron data...')
patrons_df = pd.read_csv(patrons_file, low_memory=False)
logger.info('%s patrons loaded from %s', f'{len(patrons_df):,}', patrons_file)

logger.info('Loading donor data...')
donor_df = pd.read_excel(donor_file, parse_dates=['Close Date'])
donor_df['Last Donation Date'] = pd.to_datetime(donor_df['Last Donation Date'], errors='coerce')
donor_df['Close Date']         = pd.to_datetime(donor_df['Close Date'],         errors='coerce')
# Fall back to Close Date when Last Donation Date is missing
donor_df['Last Donation Date'] = donor_df['Last Donation Date'].fillna(donor_df['Close Date'])
logger.info('%s donation records loaded', f'{len(donor_df):,}')

INFO - Loading patron data...
INFO - 22,885 patrons loaded from Patrons.csv
INFO - Loading donor data...
INFO - 4,757 donation records loaded


In [39]:
# ---------------------------------------------------------------------------
# Aggregate donations — one row per donor account
# ---------------------------------------------------------------------------
logger.info('Aggregating donations by account...')
donor_summary = (
    donor_df
    .groupby('Account Name', as_index=False)
    .agg(
        DonorAccountId    = ('Account ID',         'first'),
        LifetimeDonations = ('Amount',             'sum'),
        AverageDonation   = ('Amount',             'mean'),
        MaxDonation       = ('Amount',             'max'),
        DonationCount     = ('Amount',             'count'),
        FirstDonationDate = ('Close Date',         'min'),
        LastDonationDate  = ('Last Donation Date', 'max'),
    )
)
logger.info('%s unique donor accounts', f'{len(donor_summary):,}')

# ---------------------------------------------------------------------------
# Detect giving season per donor from individual gift dates
# Staggered: Fall = Aug–Jan (endpoint Jan 31), Spring = Feb–Jul (endpoint Jul 31)
# Gifts < MIN_SEASON_DONATION excluded (ticket add-ons, small round-ups, etc.)
# Seasons: Fall, Spring, Both (consistent in both), Mixed (no clear pattern)
# ---------------------------------------------------------------------------
_FALL_MONTHS = {8, 9, 10, 11, 12, 1}
_season_df = donor_df[donor_df['Amount'] >= MIN_SEASON_DONATION].copy()
_season_df['_season'] = _season_df['Close Date'].dt.month.apply(
    lambda m: 'Fall' if m in _FALL_MONTHS else 'Spring'
)
_season_counts = (
    _season_df.groupby(['Account Name', '_season']).size()
    .unstack(fill_value=0)
    .reindex(columns=['Fall', 'Spring'], fill_value=0)
)
_season_counts['GivingSeason'] = 'Mixed'
# Both: gifts in each season, but neither clearly dominates
_season_counts.loc[
    (_season_counts['Fall'] >= 1) & (_season_counts['Spring'] >= 1), 'GivingSeason'
] = 'Both'
# Fall/Spring override Both when one season dominates by ≥ 1.5×
_season_counts.loc[_season_counts['Fall'] > _season_counts['Spring'] * 1.5, 'GivingSeason'] = 'Fall'
_season_counts.loc[_season_counts['Spring'] > _season_counts['Fall'] * 1.5, 'GivingSeason'] = 'Spring'
donor_summary = donor_summary.merge(
    _season_counts[['GivingSeason']].reset_index(), on='Account Name', how='left'
)
donor_summary['GivingSeason'] = donor_summary['GivingSeason'].fillna('Mixed')
logger.info('Giving seasons — Fall: %s  Spring: %s  Both: %s  Mixed: %s',
            (donor_summary['GivingSeason'] == 'Fall').sum(),
            (donor_summary['GivingSeason'] == 'Spring').sum(),
            (donor_summary['GivingSeason'] == 'Both').sum(),
            (donor_summary['GivingSeason'] == 'Mixed').sum())

# ---------------------------------------------------------------------------
# Identify donor accounts with no matching patron record
# ---------------------------------------------------------------------------
matched_names    = set(patrons_df['AccountName'])
unmatched_donors = donor_summary[
    ~donor_summary['Account Name'].isin(matched_names)
].copy()
logger.info('%s donor accounts have no matching patron record (name mismatch or no ticket history)',
            f'{len(unmatched_donors):,}')

# ---------------------------------------------------------------------------
# Fuzzy-match unmatched donors against patron names
# Helps identify Account Name mismatches (whitespace, nicknames, title variants)
# ---------------------------------------------------------------------------
_patron_names = sorted(matched_names)  # sorted list for get_close_matches

def _best_match(name, candidates, cutoff=0.60):
    """Return (best_match, score) for name against candidates, or ('', 0.0)."""
    hits = difflib.get_close_matches(name, candidates, n=1, cutoff=cutoff)
    if not hits:
        return '', 0.0
    score = difflib.SequenceMatcher(None, name.lower(), hits[0].lower()).ratio()
    return hits[0], round(score, 2)

_results = unmatched_donors['Account Name'].apply(
    lambda n: pd.Series(_best_match(n, _patron_names), index=['SuggestedMatch', 'MatchScore'])
)
unmatched_donors = pd.concat([unmatched_donors, _results], axis=1)
logger.info('Fuzzy matching complete for unmatched donor accounts')

# ---------------------------------------------------------------------------
# Build name-match review dataframe
# Joins suggested matches back to patron IDs so staff can correct Salesforce
# ---------------------------------------------------------------------------
_patron_ids = patrons_df[['AccountName', 'AccountId', 'ContactId']].drop_duplicates('AccountName')
review_df = (
    unmatched_donors
    .merge(_patron_ids, left_on='SuggestedMatch', right_on='AccountName', how='left')
    .drop(columns='AccountName')
    .rename(columns={
        'Account Name':       'Donor Name (Salesforce)',
        'DonorAccountId':     'Donor Account ID (SF)',
        'SuggestedMatch':     'Suggested Patron Match',
        'MatchScore':         'Match Score',
        'AccountId':          'Patron Account ID',
        'ContactId':          'Patron Contact ID',
        'LifetimeDonations':  'Lifetime Donations',
        'AverageDonation':    'Average Donation',
        'DonationCount':      'Donation Count',
        'FirstDonationDate':  'First Donation Date',
        'LastDonationDate':   'Last Donation Date',
    })
)
_review_cols = [
    'Donor Name (Salesforce)', 'Donor Account ID (SF)',
    'Suggested Patron Match', 'Match Score',
    'Patron Account ID', 'Patron Contact ID',
    'Lifetime Donations', 'Donation Count',
    'First Donation Date', 'Last Donation Date',
]
review_df = review_df[[c for c in _review_cols if c in review_df.columns]]
review_df = review_df.sort_values('Match Score', ascending=False)

# ---------------------------------------------------------------------------
# Join to patrons (single left join)
# ---------------------------------------------------------------------------
logger.info('Merging patron and donor data...')
df = patrons_df.merge(
    donor_summary,
    left_on  = 'AccountName',
    right_on = 'Account Name',
    how      = 'left',
).drop(columns='Account Name')

df[['LifetimeDonations', 'AverageDonation', 'MaxDonation', 'DonationCount']] = (
    df[['LifetimeDonations', 'AverageDonation', 'MaxDonation', 'DonationCount']].fillna(0)
)
df['FirstDonationDate'] = pd.to_datetime(df['FirstDonationDate'], errors='coerce')
df['LastDonationDate']  = pd.to_datetime(df['LastDonationDate'],  errors='coerce')

matched   = df['DonationCount'].gt(0).sum()
unmatched = len(df) - matched
logger.info('%s patrons matched to donation records; %s with no donation history',
            f'{matched:,}', f'{unmatched:,}')


INFO - Aggregating donations by account...
INFO - 1,767 unique donor accounts
INFO - Giving seasons — Fall: 489  Spring: 157  Both: 78  Mixed: 1043
INFO - 235 donor accounts have no matching patron record (name mismatch or no ticket history)
INFO - Fuzzy matching complete for unmatched donor accounts
INFO - Merging patron and donor data...
INFO - 1,532 patrons matched to donation records; 21,353 with no donation history


In [40]:
# ---------------------------------------------------------------------------
# Derived fields
# ---------------------------------------------------------------------------
today = pd.Timestamp('today')
df['MonthsSinceLastDonation'] = (
    (today - df['LastDonationDate']).dt.days / 30
).fillna(9999)
df['IsDonor'] = df['DonationCount'] > 0
df['LatestEventDate'] = pd.to_datetime(df['LatestEventDate'], errors='coerce')

numeric_cols = ['AYM', 'Frequency', 'Lifespan', 'Regularity',
                'GrowthScore', 'Recency (Months)', 'LifetimeDonations', 'RFMScore']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)  # guard against inf values

# Derived scoring inputs used by _propensity_score
df['_UpgradeRatio']      = df['AYM'] / df['AverageDonation'].replace(0, np.nan)
df['_DormantRecency']    = 1 / df['MonthsSinceLastDonation'].clip(lower=1)
df['_LogDonationCount']  = np.log1p(df['DonationCount'])
df['_IsChorus']          = df['ChorusMember'].astype(float)
_patron_rank = {'officer': 4, 'board': 3, 'ex-board': 2, 'corporator': 1, 'patron': 0}
df['_PatronRank']        = df['PatronStatus'].map(_patron_rank).fillna(0).astype(float)
_subscriber_rank = {'current': 2, 'previous': 1, 'never': 0}
df['_SubscriberRank']    = df['Subscriber'].map(_subscriber_rank).fillna(0).astype(float)

# Seasonal due score — replaces _DonationFreshness for active donor tranches
# Staggered season endpoints: Fall = Jan 31, Spring = Jul 31
def _last_gift_season_endpoint(last_date, giving_season):
    """Return the endpoint of the season that contains the last gift date."""
    y, m = last_date.year, last_date.month
    if giving_season == 'Fall':
        return pd.Timestamp(y + 1, 1, 31) if m >= 8 else pd.Timestamp(y, 1, 31)
    else:  # Spring
        if 2 <= m <= 7:  return pd.Timestamp(y, 7, 31)
        elif m >= 8:     return pd.Timestamp(y, 7, 31)
        else:            return pd.Timestamp(y - 1, 7, 31)   # Jan is off-season for Spring

def _compute_donor_due(row, today):
    giving_season  = row.get('GivingSeason', 'Mixed')
    last_date      = row.get('LastDonationDate')
    months_since   = row.get('MonthsSinceLastDonation', 9999)

    if pd.isna(last_date) or giving_season in ('Mixed', 'Both'):
        # No single season — use annual cadence: 0 if gave within a year, else years elapsed
        years_missed = int(months_since // 12) if months_since < 9999 else np.nan
        return pd.Series({'SeasonsMissed': years_missed,
                          '_DonorDueScore': max(0.0, 1.0 - months_since / 18.0)})

    ep_month  = 1 if giving_season == 'Fall' else 7
    last_ep   = _last_gift_season_endpoint(last_date, giving_season)

    missed = sum(
        1 for yr in range(last_ep.year, today.year + 2)
        if last_ep < pd.Timestamp(yr, ep_month, 31) <= today
    )

    if missed == 0:
        in_season = (giving_season == 'Fall'   and (today.month >= 8 or today.month == 1)) or \
                    (giving_season == 'Spring' and 2 <= today.month <= 7)
        score = 0.7 if in_season else 0.1
    elif missed == 1:
        score = 0.9
    else:
        score = 1.0

    return pd.Series({'SeasonsMissed': float(missed), '_DonorDueScore': score})

_due = df.apply(lambda row: _compute_donor_due(row, today), axis=1)
df['SeasonsMissed']  = _due['SeasonsMissed']
df['_DonorDueScore'] = _due['_DonorDueScore']


def _propensity_score(frame, components):
    """Return a 0–100 score: weighted sum of min-max normalized inputs.

    NaN values are replaced with the tranche median before normalization.
    When all values in a component are equal, that component contributes 0.5 × weight.
    Missing columns are skipped with a warning — their weight is redistributed
    proportionally across present components.
    """
    if frame.empty:
        return pd.Series(dtype=float)

    present   = [(col, w) for col, w in components if col in frame.columns]
    missing   = [(col, w) for col, w in components if col not in frame.columns]
    for col, _ in missing:
        logger.warning('_propensity_score: column %r not in data — skipped. '
                       'Re-run MWSalesSumm to generate updated Patrons.csv.', col)

    # Redistribute weight of missing columns proportionally
    total_w   = sum(w for _, w in present) or 1.0
    score     = pd.Series(0.0, index=frame.index)
    for col, weight in present:
        s = frame[col].copy()
        s = s.fillna(s.median() if s.notna().any() else 0.0)
        lo, hi = s.min(), s.max()
        if hi > lo:
            s = (s - lo) / (hi - lo)
        else:
            s = pd.Series(0.5, index=frame.index)
        score += (weight / total_w) * s
    return (score * 100).round(1)


In [41]:
# ---------------------------------------------------------------------------
# Tranche classification (mutually exclusive — each patron appears at most once)
# ---------------------------------------------------------------------------
logger.info('Classifying tranches...')

# Tranche 1: Major Patrons — high-value current donors, upgrade ask
major_donors = df[
    df['IsDonor'] &
    (df['MonthsSinceLastDonation'] <= ACTIVE_DONATION_MONTHS) &
    (df['MaxDonation'] >= MAJOR_GIFT_MIN)
].copy()
major_donors['DonorPropensityScore'] = _propensity_score(major_donors, T1_WEIGHTS)
major_donors = major_donors.sort_values('DonorPropensityScore', ascending=False)
classified_ids = set(major_donors['AccountId'])
logger.info('  Major Patrons:                %s', f'{len(major_donors):,}')

# Tranche 2: Growth Prospects — active donors with strong attendance, upgrade ask
# Best/High/Upsell segments, recently active, donation within 18 months
growth_prospects = df[
    ~df['AccountId'].isin(classified_ids) &
    df['IsDonor'] &
    (df['MonthsSinceLastDonation'] <= ACTIVE_DONATION_MONTHS) &
    df['Segment'].isin(['Best', 'High', 'Upsell']) &
    (df['Recency (Months)'] <= PRIME_RECENCY_MONTHS)
].copy()
growth_prospects['DonorPropensityScore'] = _propensity_score(growth_prospects, T2_WEIGHTS)
growth_prospects = growth_prospects.sort_values('DonorPropensityScore', ascending=False)
classified_ids |= set(growth_prospects['AccountId'])
logger.info('  Growth Prospects:            %s', f'{len(growth_prospects):,}')

# Tranche 3: Active Donors — Renew — all remaining donors who gave recently
# No segment filter: preserves donors who may show as attendance-Lapsed post-COVID
active_donors = df[
    ~df['AccountId'].isin(classified_ids) &
    df['IsDonor'] &
    (df['MonthsSinceLastDonation'] <= ACTIVE_DONATION_MONTHS)
].copy()
active_donors['DonorPropensityScore'] = _propensity_score(active_donors, T3_WEIGHTS)
active_donors = active_donors.sort_values('DonorPropensityScore', ascending=False)
classified_ids |= set(active_donors['AccountId'])
logger.info('  Active Donors — Renew:       %s', f'{len(active_donors):,}')

# Tranche 4: Dormant Donors — Reactivate
# Gave before but not recently; still attending (not Lapsed or One&Done)
dormant_donors = df[
    ~df['AccountId'].isin(classified_ids) &
    df['IsDonor'] &
    (df['MonthsSinceLastDonation'] > ACTIVE_DONATION_MONTHS) &
    ~df['Segment'].isin(['Lapsed', 'One&Done'])
].copy()
dormant_donors['DonorPropensityScore'] = _propensity_score(dormant_donors, T4_WEIGHTS)
dormant_donors = dormant_donors.sort_values('DonorPropensityScore', ascending=False)
classified_ids |= set(dormant_donors['AccountId'])
logger.info('  Dormant Donors — Reactivate: %s', f'{len(dormant_donors):,}')

# Tranche 5: Prime Non-Donor Prospects — first gift ask, high confidence
# Best/High/Upsell segments, recently active; New/Re-engaged/Come Again excluded
prime_prospects = df[
    ~df['AccountId'].isin(classified_ids) &
    ~df['IsDonor'] &
    df['Segment'].isin(['Best', 'High', 'Upsell']) &
    (df['Recency (Months)'] <= PRIME_RECENCY_MONTHS)
].copy()
prime_prospects['DonorPropensityScore'] = _propensity_score(prime_prospects, T5_WEIGHTS)
prime_prospects = prime_prospects.sort_values('DonorPropensityScore', ascending=False)
classified_ids |= set(prime_prospects['AccountId'])
logger.info('  Prime Non-Donor Prospects:   %s', f'{len(prime_prospects):,}')

# Lapsed Donors — donors whose attendance segment is Lapsed or One&Done (review only)
lapsed_donors = df[
    df['IsDonor'] &
    ~df['AccountId'].isin(classified_ids)
].copy()
lapsed_donors['DonorPropensityScore'] = _propensity_score(lapsed_donors, T6_WEIGHTS)
lapsed_donors = lapsed_donors.sort_values('DonorPropensityScore', ascending=False)
logger.info('  Lapsed Donors (review only): %s', f'{len(lapsed_donors):,}')

total = (len(major_donors) + len(growth_prospects) + len(active_donors) +
         len(dormant_donors) + len(prime_prospects))
logger.info('Total actionable: %s of %s patrons', f'{total:,}', f'{len(df):,}')


INFO - Classifying tranches...
INFO -   Major Patrons:                86
INFO -   Growth Prospects:            208
INFO -   Active Donors — Renew:       718
INFO -   Dormant Donors — Reactivate: 489
INFO -   Prime Non-Donor Prospects:   148
INFO -   Lapsed Donors (review only): 31
INFO - Total actionable: 1,649 of 22,885 patrons


In [42]:
# ---------------------------------------------------------------------------
# Output helpers
# ---------------------------------------------------------------------------
PII_COLS = {'Account Name', 'First Name', 'Last Name', 'Email Address'}

COL_MAP = {
    'AccountName':             'Account Name',
    'AccountId':               'Account ID',
    'FirstName':               'First Name',
    'LastName':                'Last Name',
    'Email':                   'Email Address',
    'DonorPropensityScore':    'Propensity Score',
    'Segment':                 'Customer Segment',
    'RFMScore':                'RFM Score',
    'Recency (Months)':        'Recency (Months)',
    'Frequency':               'Frequency',
    'FullPriceRate':           'Full Price Rate',
    'AYM':                     'Avg Yearly Monetary',
    'Lifespan':                'Customer Lifespan',
    'GrowthScore':             'Growth Score',
    'Regularity':              'Regularity',
    'ChorusMember':            'Chorus Member',
    'PatronStatus':            'Patron Status',
    'Subscriber':              'Subscriber',
    'IsDonor':                 'Is Donor',
    'LifetimeDonations':       'Lifetime Donations',
    'AverageDonation':         'Average Donation',
    'DonationCount':           'Donation Count',
    'FirstDonationDate':       'First Donation Date',
    'LastDonationDate':        'Last Donation Date',
    'MonthsSinceLastDonation': 'Months Since Last Donation',
    'LatestEventDate':         'Last Event Date',
    'GivingSeason':            'Giving Season',
    'SeasonsMissed':           'Seasons Missed',
    'PreferredEventGenre':     'Favorite Genre',
    'PreferredEventClass':     'Favorite Class',
    'RegionAssignment':        'Geo Region',
}

CURRENCY_COLS = {'Lifetime Donations', 'Average Donation', 'Avg Yearly Monetary'}
DATE_COLS     = {'First Donation Date', 'Last Donation Date'}


def _prepare(frame):
    """Select and rename columns, keeping only those present in the frame."""
    available = {k: v for k, v in COL_MAP.items() if k in frame.columns}
    return frame[list(available.keys())].rename(columns=available)


def _write_sheet(writer, data, sheet_name, currency_cols=None, date_cols=None):
    """Write a DataFrame to an Excel sheet with formatting."""
    if currency_cols is None:
        currency_cols = CURRENCY_COLS
    if date_cols is None:
        date_cols = DATE_COLS
    out = data if isinstance(data, pd.DataFrame) else _prepare(data)
    out.to_excel(writer, sheet_name=sheet_name, index=False)
    ws   = writer.sheets[sheet_name]
    book = writer.book
    last_col = len(out.columns) - 1
    ws.freeze_panes(1, 0)
    ws.autofilter(0, 0, len(out), last_col)

    header_fmt = book.add_format({'text_wrap': True, 'bold': True, 'valign': 'top'})
    ws.set_row(0, 45)
    for col_num, col_name in enumerate(out.columns):
        ws.write(0, col_num, col_name, header_fmt)

    _auto_size = {'Account Name', 'First Donation Date', 'Last Donation Date'}
    for i, col in enumerate(out.columns):
        if col in _auto_size:
            col_width = max(out[col].astype(str).map(len).max(), len(col)) + 2
        else:
            col_width = 6
        fmt = None
        if col in date_cols:
            fmt = book.add_format({'num_format': 'mm/dd/yyyy'})
        elif col in currency_cols:
            fmt = book.add_format({'num_format': '$#,##0'})
        elif out[col].dtype.kind in 'fi':
            fmt = book.add_format({'num_format': '0.0'})
        ws.set_column(i, i, col_width, fmt)


# ---------------------------------------------------------------------------
# Summary report helpers
# ---------------------------------------------------------------------------
_SUMMARY_BASE_COLS = [
    ('AccountName',          'Name'),
    ('DonorPropensityScore', 'Priority Score'),
    ('Frequency',            'Events Attended'),
    ('FullPriceRate',        'Full Price Rate'),
    ('AYM',                  'Avg Yearly Spend'),
    ('Lifespan',             'Years as Patron'),
    ('LatestEventDate',      'Last Event Date'),
    ('PreferredEventGenre',  'Favorite Genre'),
    ('PreferredEventClass',  'Favorite Class'),
    ('ChorusMember',         'In Chorus'),
    ('Subscriber',           'Subscriber'),
    ('PatronStatus',         'Board Role'),
    ('RegionAssignment',     'Region'),
    ('Email',                'Email'),
]
_SUMMARY_SEASON_COLS = [
    ('GivingSeason',    'Giving Season'),
    ('SeasonsMissed',   'Seasons Missed'),
    ('LastDonationDate','Last Gift Date'),
]
_SUMMARY_DONOR_COLS = [
    ('LifetimeDonations', 'Lifetime Giving'),
    ('DonationCount',     'Total Gifts'),
]
_SUMMARY_PII_SRC  = {'AccountName', 'Email'}
_SUMMARY_CURRENCY = {'Avg Yearly Spend', 'Lifetime Giving'}
_SUMMARY_DATE     = {'Last Gift Date', 'Last Event Date'}
_SUMMARY_AUTO     = {'Name', 'Email'}
_SUMMARY_WIDE     = {'Favorite Genre', 'Favorite Class', 'Region', 'Board Role', 'Subscriber', 'Giving Season', 'Full Price Rate'}

_TRANCHE_COLORS = {
    'Major Patrons':                {'tab': '#C9A800', 'header': '#FFF2CC'},
    'Growth Prospects':            {'tab': '#00813A', 'header': '#E2EFDA'},
    'Active Donors - Renew':       {'tab': '#2E5FAA', 'header': '#DCE6F1'},
    'Dormant Donors - Reactivate': {'tab': '#C55A11', 'header': '#FCE4D6'},
    'Prime Non-Donor Prospects':   {'tab': '#6B2E8C', 'header': '#EAD1DC'},
    'Lapsed Donors - Review':      {'tab': '#595959', 'header': '#F2F2F2'},
}
_TRANCHE_INFO = [
    ('Major Patrons',               'Upgrade ask',
     'High-value current donors (avg yearly spend ≥ $1,500 or lifetime giving ≥ $5,000). '
     'Priority candidates for major gift conversations and upgrade asks.'),
    ('Growth Prospects',           'Upgrade ask',
     'Active donors in our best attendance segments showing strong growth. '
     'Well-positioned for an increased giving ask.'),
    ('Active Donors - Renew',      'Renew',
     'Donors who gave within the last 18 months. Focus on securing renewals before they lapse.'),
    ('Dormant Donors - Reactivate','Reactivate',
     'Donors who have lapsed in giving but are still attending events. '
     'A warm reactivation ask is appropriate.'),
    ('Prime Non-Donor Prospects',  'First gift',
     'Highly engaged non-donors in our best attendance segments. '
     'Strong candidates for a first-gift conversation.'),
    ('Lapsed Donors - Review',     'Review only',
     'Donors whose event attendance has also lapsed. Review individually before any outreach.'),
]


def _write_summary_sheet(writer, frame, sheet_name, is_donor_tranche, drop_pii=False):
    """Write one summary sheet: top SUMMARY_TOP_N visible by default, all rows present."""
    colors = _TRANCHE_COLORS.get(sheet_name, {'tab': '#2E5FAA', 'header': '#DCE6F1'})
    book   = writer.book

    col_map = list(_SUMMARY_BASE_COLS)
    if is_donor_tranche:
        # Insert season cols right after Priority Score (index 1), then append donation cols at end
        col_map = col_map[:2] + list(_SUMMARY_SEASON_COLS) + col_map[2:] + list(_SUMMARY_DONOR_COLS)
    if drop_pii:
        col_map = [(s, d) for s, d in col_map if s not in _SUMMARY_PII_SRC]

    available = [(s, d) for s, d in col_map if s in frame.columns]
    out = frame[[s for s, _ in available]].copy()
    out = out.rename(columns=dict(available))

    if 'In Chorus' in out.columns:
        out['In Chorus'] = out['In Chorus'].map({True: 'Yes', False: 'No', 1: 'Yes', 0: 'No'}).fillna('No')
    if 'Subscriber' in out.columns:
        out['Subscriber'] = out['Subscriber'].map({'current': 'Current', 'previous': 'Past', 'never': ''}).fillna('')
    if 'Board Role' in out.columns:
        out['Board Role'] = out['Board Role'].apply(
            lambda v: '' if pd.isna(v) or str(v).lower() == 'patron' else str(v).capitalize()
        )

    out.insert(0, 'Rank', range(1, len(out) + 1))
    out.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]

    ws.set_tab_color(colors['tab'])
    ws.freeze_panes(1, 0)
    ws.autofilter(0, 0, len(out), len(out.columns) - 1)
    ws.filter_column(0, 'x <= %d' % SUMMARY_TOP_N)
    for row_num in range(SUMMARY_TOP_N + 1, len(out) + 1):
        ws.set_row(row_num, None, None, {'hidden': True})

    hdr_fmt = book.add_format({
        'text_wrap': True, 'bold': True, 'valign': 'top',
        'bg_color': colors['header'], 'border': 1,
    })
    ws.set_row(0, 45)
    for col_num, col_name in enumerate(out.columns):
        ws.write(0, col_num, col_name, hdr_fmt)

    # Conditional formatting: Priority Score — white → light blue → dark blue gradient
    if 'Priority Score' in out.columns:
        sc = list(out.columns).index('Priority Score')
        ws.conditional_format(1, sc, len(out), sc, {
            'type': '3_color_scale',
            'min_color': '#FFFFFF', 'mid_color': '#FFEB84', 'max_color': '#63BE7B',
        })

    # Conditional formatting: Seasons Missed — green (0), amber (1), red (2+)
    if 'Seasons Missed' in out.columns:
        sm = list(out.columns).index('Seasons Missed')
        ws.conditional_format(1, sm, len(out), sm, {
            'type': 'cell', 'criteria': '==', 'value': 0,
            'format': book.add_format({'bg_color': '#C6EFCE', 'font_color': '#276221'}),
        })
        ws.conditional_format(1, sm, len(out), sm, {
            'type': 'cell', 'criteria': '==', 'value': 1,
            'format': book.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C6500'}),
        })
        ws.conditional_format(1, sm, len(out), sm, {
            'type': 'cell', 'criteria': '>=', 'value': 2,
            'format': book.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'}),
        })

    for i, col in enumerate(out.columns):
        if col in _SUMMARY_AUTO:
            col_width = max(out[col].astype(str).map(len).max(), len(col)) + 2
        elif col in _SUMMARY_WIDE:
            col_width = 14
        elif col in _SUMMARY_DATE:
            col_width = 13
        else:
            col_width = 8
        fmt = None
        if col in _SUMMARY_DATE:
            fmt = book.add_format({'num_format': 'mm/dd/yyyy'})
        elif col in _SUMMARY_CURRENCY:
            fmt = book.add_format({'num_format': '$#,##0'})
        elif col == 'Priority Score':
            fmt = book.add_format({'num_format': '0.0'})
        elif col == 'Full Price Rate':
            fmt = book.add_format({'num_format': '0%'})
        elif col == 'Rank':
            fmt = book.add_format({'num_format': '0'})
        elif out[col].dtype.kind in 'fi':
            fmt = book.add_format({'num_format': '0'})
        ws.set_column(i, i, col_width, fmt)


def _write_how_to_use(writer):
    """Write a plain-English guide sheet for non-technical staff."""
    book = writer.book
    ws   = book.add_worksheet('How to Use')
    ws.set_tab_color('#1F497D')
    ws.set_column(0, 0, 22)
    ws.set_column(1, 1, 72)

    title_fmt   = book.add_format({'bold': True, 'font_size': 14, 'font_color': '#1F497D', 'bottom': 2})
    section_fmt = book.add_format({'bold': True, 'bg_color': '#D9E1F2', 'border': 1, 'valign': 'vcenter'})
    label_fmt   = book.add_format({'bold': True, 'valign': 'top', 'text_wrap': True})
    body_fmt    = book.add_format({'text_wrap': True, 'valign': 'top'})

    def row_write(r, label, body, row_height=None):
        ws.write(r, 0, label, label_fmt)
        ws.write(r, 1, body,  body_fmt)
        if row_height:
            ws.set_row(r, row_height)

    r = 0
    ws.merge_range(r, 0, r, 1, 'Patron Classification Report — How to Use', title_fmt)
    ws.set_row(r, 24)
    r += 2

    ws.merge_range(r, 0, r, 1, 'TABS — What each group means and what action to take', section_fmt)
    ws.set_row(r, 18)
    r += 1
    tabs = [
        ('Major Patrons',                'Currently giving at high levels ($1,500+/yr or $5,000+ lifetime). '
                                        'Priority for major gift conversations and upgrade asks.'),
        ('Growth Prospects',            'Active donors in our best attendance segments with strong growth. '
                                        'Well-positioned for an increased giving ask.'),
        ('Active Donors - Renew',       'Gave within the last 18 months. Focus on securing renewal '
                                        'before they lapse.'),
        ('Dormant Donors - Reactivate', 'Previously donated but have stopped giving. Still attending events. '
                                        'A warm reactivation ask is appropriate.'),
        ('Prime Non-Donor Prospects',   'Highly engaged attendees who have never donated. '
                                        'Ideal first-gift candidates.'),
        ('Lapsed Donors - Review',      'Both giving and attendance have lapsed. '
                                        'Review individually before any outreach.'),
    ]
    for label, desc in tabs:
        row_write(r, label, desc, row_height=30)
        r += 1
    r += 1

    ws.merge_range(r, 0, r, 1, 'COLUMNS — What each column means', section_fmt)
    ws.set_row(r, 18)
    r += 1
    cols = [
        ('Priority Score',   '1–100 ranking within this group. Higher = more likely to give now. '
                             'Sort by this first when deciding who to contact.'),
        ('Giving Season',    'Whether this patron tends to give in Fall (Aug–Jan) or Spring (Feb–Jul). '
                             '"Mixed" means no clear pattern.'),
        ('Seasons Missed',   'How many giving seasons have passed since their last gift. '
                             'Fall/Spring donors count 6-month seasons; Mixed donors count years. '
                             'Green = current (0), Amber = one missed, Red = two or more — highest reactivation priority.'),
        ('Last Gift Date',   'Date of their most recent donation.'),
        ('Last Event Date',  'Date they last attended a Music Worcester event.'),
        ('Events Attended',  'Total number of events attended across all years.'),
        ('Avg Yearly Spend', 'Average amount spent on tickets per year (not including donations).'),
        ('Years as Patron',  'How long they have been buying tickets from Music Worcester.'),
        ('Lifetime Giving',  'Total amount donated to Music Worcester over all years.'),
        ('Total Gifts',      'Number of separate donations made.'),
        ('Favorite Genre',   'The type of music they attend most (Classical, Pops, World, etc.).'),
        ('Favorite Class',   'The event tier they prefer most (Headliner, Signature, Community, etc.).'),
        ('In Chorus',        'Yes = current Worcester Chorus member.'),
        ('Full Price Rate',  'Proportion of tickets purchased at full price (excludes Chorus, EBT, '
                             'and exchange discounts). Higher = stronger indicator of giving capacity.'),
        ('Subscriber',       'Current = active subscriber; Past = former subscriber.'),
        ('Board Role',       'Board member, corporator, or officer (blank = general patron).'),
        ('Region',           'Geographic area based on mailing address.'),
    ]
    for label, desc in cols:
        row_write(r, label, desc, row_height=40)
        r += 1
    r += 1

    ws.merge_range(r, 0, r, 1, 'GETTING STARTED', section_fmt)
    ws.set_row(r, 18)
    r += 1
    ws.merge_range(r, 0, r, 1,
        '1. Choose the tab for the outreach you are planning.\n'
        '2. The top 50 rows are shown by default, ranked by Priority Score. '
        'To see all patrons, clear the filter on the Rank column.\n'
        '3. For donors: check Seasons Missed first — red rows are most overdue.\n'
        '4. Use Last Event Date to confirm the patron is still active before reaching out.\n'
        '5. In Chorus, Board Role, and Subscriber status are strong signals — prioritize them '
        'when outreach capacity is limited.',
        body_fmt)
    ws.set_row(r, 90)


def _write_summary_overview(writer, tranche_counts):
    """Write an Overview sheet with tranche descriptions and counts."""
    book = writer.book
    rows = [(name, tranche_counts[name], action, desc)
            for name, action, desc in _TRANCHE_INFO]
    overview = pd.DataFrame(rows, columns=['Tranche', 'Total Count', 'Action', 'Description'])
    overview.to_excel(writer, sheet_name='Overview', index=False)
    ws = writer.sheets['Overview']
    ws.set_tab_color('#1F497D')

    hdr_fmt = book.add_format({
        'bold': True, 'valign': 'vcenter', 'text_wrap': True,
        'bg_color': '#1F497D', 'font_color': '#FFFFFF', 'border': 1,
    })
    ws.set_row(0, 30)
    for col_num, col_name in enumerate(['Tranche', 'Total Count', 'Action', 'Description']):
        ws.write(0, col_num, col_name, hdr_fmt)

    ws.set_column(0, 0, 28)
    ws.set_column(1, 1, 12)
    ws.set_column(2, 2, 14)
    ws.set_column(3, 3, 60)

    for row_idx, (tranche_name, _, _, _) in enumerate(rows, start=1):
        bg = _TRANCHE_COLORS.get(tranche_name, {}).get('header', '#FFFFFF')
        row_fmt = book.add_format({'bg_color': bg, 'text_wrap': True, 'valign': 'vcenter', 'border': 1})
        ws.set_row(row_idx, 45, row_fmt)
        for col_num, val in enumerate(rows[row_idx - 1]):
            ws.write(row_idx, col_num, val, row_fmt)


_CAMPAIGN_TRANCHE_ORDER = [
    'Major Patrons', 'Growth Prospects', 'Active Donors - Renew',
    'Dormant Donors - Reactivate', 'Prime Non-Donor Prospects',
]
_next_campaign = 'Fall' if today.month in _FALL_MONTHS else 'Spring'


def _build_campaign_frame(season):
    """Return a combined DataFrame of all patrons targeted for one campaign season."""
    subsets = []
    donor_tranches = [
        (major_donors,    'Major Patrons'),
        (growth_prospects,'Growth Prospects'),
        (active_donors,   'Active Donors - Renew'),
        (dormant_donors,  'Dormant Donors - Reactivate'),
    ]
    for frame, name in donor_tranches:
        mask = frame['GivingSeason'].isin([season, 'Both'])
        if season == _next_campaign:
            mask = mask | (frame['GivingSeason'] == 'Mixed')
        sub = frame[mask].copy()
        sub['_Tranche'] = name
        sub['_InBoth']  = sub['GivingSeason'].isin(['Both']) | (sub['GivingSeason'] == 'Mixed')
        subsets.append(sub)

    t5 = prime_prospects.copy()
    t5['_Tranche'] = 'Prime Non-Donor Prospects'
    t5['_InBoth']  = True
    subsets.append(t5)

    combined = pd.concat(subsets, ignore_index=True)
    combined['_TrancheRank'] = combined['_Tranche'].map(
        {t: i for i, t in enumerate(_CAMPAIGN_TRANCHE_ORDER)}
    )
    combined = combined.sort_values(
        ['_TrancheRank', 'DonorPropensityScore'],
        ascending=[True, False],
    ).drop(columns=['_TrancheRank'])
    return combined


def _write_campaign_sheet(writer, frame, sheet_name, drop_pii=False):
    """Write one campaign sheet (Fall or Spring) with all targeted patrons."""
    book   = writer.book
    colors = {'tab': '#1F497D', 'header': '#D9E1F2'}

    col_map = (
        [('_Tranche',            'Tranche')]
        + [('AccountName',          'Name')]
        + [('DonorPropensityScore', 'Priority Score')]
        + list(_SUMMARY_SEASON_COLS)
        + [
            ('Frequency',            'Events Attended'),
            ('FullPriceRate',        'Full Price Rate'),
            ('AYM',                  'Avg Yearly Spend'),
            ('Lifespan',             'Years as Patron'),
            ('LatestEventDate',      'Last Event Date'),
            ('PreferredEventGenre',  'Favorite Genre'),
            ('PreferredEventClass',  'Favorite Class'),
            ('ChorusMember',         'In Chorus'),
            ('Subscriber',           'Subscriber'),
            ('PatronStatus',         'Board Role'),
            ('RegionAssignment',     'Region'),
            ('Email',                'Email'),
            ('_InBoth',              'In Both Campaigns'),
        ]
        + list(_SUMMARY_DONOR_COLS)
    )

    if drop_pii:
        col_map = [(s, d) for s, d in col_map if s not in _SUMMARY_PII_SRC]

    available = [(s, d) for s, d in col_map if s in frame.columns]
    out = frame[[s for s, _ in available]].copy()
    out = out.rename(columns=dict(available))

    if 'In Chorus' in out.columns:
        out['In Chorus'] = out['In Chorus'].map({True: 'Yes', False: 'No', 1: 'Yes', 0: 'No'}).fillna('No')
    if 'Subscriber' in out.columns:
        out['Subscriber'] = out['Subscriber'].map({'current': 'Current', 'previous': 'Past', 'never': ''}).fillna('')
    if 'Board Role' in out.columns:
        out['Board Role'] = out['Board Role'].apply(
            lambda v: '' if pd.isna(v) or str(v).lower() == 'patron' else str(v).capitalize()
        )
    if 'In Both Campaigns' in out.columns:
        out['In Both Campaigns'] = out['In Both Campaigns'].map({True: 'Yes', False: ''}).fillna('')

    out.insert(0, 'Rank', range(1, len(out) + 1))
    out.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]

    ws.set_tab_color(colors['tab'])
    ws.freeze_panes(1, 0)
    ws.autofilter(0, 0, len(out), len(out.columns) - 1)

    hdr_fmt = book.add_format({
        'text_wrap': True, 'bold': True, 'valign': 'top',
        'bg_color': colors['header'], 'border': 1,
    })
    ws.set_row(0, 45)
    for col_num, col_name in enumerate(out.columns):
        ws.write(0, col_num, col_name, hdr_fmt)

    if 'Priority Score' in out.columns:
        sc = list(out.columns).index('Priority Score')
        ws.conditional_format(1, sc, len(out), sc, {
            'type': '3_color_scale',
            'min_color': '#FFFFFF', 'mid_color': '#FFEB84', 'max_color': '#63BE7B',
        })

    if 'Seasons Missed' in out.columns:
        sm = list(out.columns).index('Seasons Missed')
        ws.conditional_format(1, sm, len(out), sm, {
            'type': 'cell', 'criteria': '==', 'value': 0,
            'format': book.add_format({'bg_color': '#C6EFCE', 'font_color': '#276221'}),
        })
        ws.conditional_format(1, sm, len(out), sm, {
            'type': 'cell', 'criteria': '==', 'value': 1,
            'format': book.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C6500'}),
        })
        ws.conditional_format(1, sm, len(out), sm, {
            'type': 'cell', 'criteria': '>=', 'value': 2,
            'format': book.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'}),
        })

    for i, col in enumerate(out.columns):
        if col in _SUMMARY_AUTO:
            col_width = max(out[col].astype(str).map(len).max(), len(col)) + 2
        elif col in _SUMMARY_WIDE or col == 'Tranche':
            col_width = 16
        elif col in _SUMMARY_DATE:
            col_width = 13
        else:
            col_width = 8
        fmt = None
        if col in _SUMMARY_DATE:
            fmt = book.add_format({'num_format': 'mm/dd/yyyy'})
        elif col in _SUMMARY_CURRENCY:
            fmt = book.add_format({'num_format': '$#,##0'})
        elif col == 'Priority Score':
            fmt = book.add_format({'num_format': '0.0'})
        elif col == 'Full Price Rate':
            fmt = book.add_format({'num_format': '0%'})
        elif col == 'Rank':
            fmt = book.add_format({'num_format': '0'})
        elif out[col].dtype.kind in 'fi':
            fmt = book.add_format({'num_format': '0'})
        ws.set_column(i, i, col_width, fmt)


def _write_campaign(writer, drop_pii=False):
    """Write Fall and Spring campaign sheets."""
    fall_df   = _build_campaign_frame('Fall')
    spring_df = _build_campaign_frame('Spring')
    _write_campaign_sheet(writer, fall_df,   'Fall Campaign',   drop_pii)
    _write_campaign_sheet(writer, spring_df, 'Spring Campaign', drop_pii)


def _write_summary(writer, drop_pii=False):
    """Write How to Use + Overview + all tranche summary sheets."""
    counts = {
        'Major Patrons':                len(major_donors),
        'Growth Prospects':            len(growth_prospects),
        'Active Donors - Renew':       len(active_donors),
        'Dormant Donors - Reactivate': len(dormant_donors),
        'Prime Non-Donor Prospects':   len(prime_prospects),
        'Lapsed Donors - Review':      len(lapsed_donors),
    }
    _write_how_to_use(writer)
    _write_summary_overview(writer, counts)
    _write_summary_sheet(writer, major_donors,    'Major Patrons',                is_donor_tranche=True,  drop_pii=drop_pii)
    _write_summary_sheet(writer, growth_prospects,'Growth Prospects',            is_donor_tranche=True,  drop_pii=drop_pii)
    _write_summary_sheet(writer, active_donors,   'Active Donors - Renew',       is_donor_tranche=True,  drop_pii=drop_pii)
    _write_summary_sheet(writer, dormant_donors,  'Dormant Donors - Reactivate', is_donor_tranche=True,  drop_pii=drop_pii)
    _write_summary_sheet(writer, prime_prospects, 'Prime Non-Donor Prospects',   is_donor_tranche=False, drop_pii=drop_pii)
    _write_summary_sheet(writer, lapsed_donors,   'Lapsed Donors - Review',      is_donor_tranche=True,  drop_pii=drop_pii)


# ---------------------------------------------------------------------------
# Write output
# ---------------------------------------------------------------------------
_DATE_FMT = {'datetime_format': 'mm/dd/yyyy', 'date_format': 'mm/dd/yyyy'}

def _write_tranches(writer, drop_pii=False):
    """Write all tranche sheets to an ExcelWriter. If drop_pii, omit PII columns."""
    def prep(frame):
        out = _prepare(frame)
        if drop_pii:
            out = out.drop(columns=[c for c in PII_COLS if c in out.columns])
        return out

    _write_sheet(writer, prep(major_donors),    'Major Patrons')
    _write_sheet(writer, prep(growth_prospects),'Growth Prospects')
    _write_sheet(writer, prep(active_donors),   'Active Donors - Renew')
    _write_sheet(writer, prep(dormant_donors),  'Dormant Donors - Reactivate')
    _write_sheet(writer, prep(prime_prospects), 'Prime Non-Donor Prospects')
    _write_sheet(writer, prep(lapsed_donors),   'Lapsed Donors - Review')
    if not drop_pii:
        _write_sheet(writer,
                     unmatched_donors.rename(columns={
                         'LifetimeDonations': 'Lifetime Donations',
                         'AverageDonation':   'Average Donation',
                         'DonationCount':     'Donation Count',
                         'FirstDonationDate': 'First Donation Date',
                         'LastDonationDate':  'Last Donation Date',
                     }),
                     'Donors - No Attendance Match',
                     currency_cols={'Lifetime Donations', 'Average Donation'},
                     date_cols={'First Donation Date', 'Last Donation Date'})

logger.info('Writing %s...', output_file)
with pd.ExcelWriter(output_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_tranches(writer, drop_pii=False)
logger.info('Done. Output written to: %s', output_file)

logger.info('Writing %s...', anon_file)
with pd.ExcelWriter(anon_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_tranches(writer, drop_pii=True)
logger.info('Done. Anonymized output written to: %s', anon_file)

logger.info('Writing %s...', review_file)
with pd.ExcelWriter(review_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_sheet(
        writer, review_df, 'Donor Name Match Review',
        currency_cols={'Lifetime Donations', 'Average Donation'},
        date_cols={'First Donation Date', 'Last Donation Date'},
    )
logger.info('Name match review written to: %s', review_file)

logger.info('Writing %s...', summary_file)
with pd.ExcelWriter(summary_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_summary(writer, drop_pii=False)
logger.info('Done. Summary written to: %s', summary_file)

logger.info('Writing %s...', anon_summary_file)
with pd.ExcelWriter(anon_summary_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_summary(writer, drop_pii=True)
logger.info('Done. Anonymized summary written to: %s', anon_summary_file)

logger.info('Writing %s... (next campaign: %s)', campaign_file, _next_campaign)
with pd.ExcelWriter(campaign_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_campaign(writer, drop_pii=False)
logger.info('Done. Campaign lists written to: %s', campaign_file)

logger.info('Writing %s...', anon_campaign_file)
with pd.ExcelWriter(anon_campaign_file, engine='xlsxwriter', **_DATE_FMT) as writer:
    _write_campaign(writer, drop_pii=True)
logger.info('Done. Anonymized campaign lists written to: %s', anon_campaign_file)


INFO - Writing Donor_Classification.xlsx...
INFO - Done. Output written to: Donor_Classification.xlsx
INFO - Writing Donor_Classification_Anon.xlsx...
INFO - Done. Anonymized output written to: Donor_Classification_Anon.xlsx
INFO - Writing DonorNameMatchReview.xlsx...
INFO - Name match review written to: DonorNameMatchReview.xlsx
INFO - Writing Donor_Classification_Summary.xlsx...
INFO - Done. Summary written to: Donor_Classification_Summary.xlsx
INFO - Writing Donor_Classification_Summary_Anon.xlsx...
INFO - Done. Anonymized summary written to: Donor_Classification_Summary_Anon.xlsx
